# WWR Segmentation — Google Colab (A100)

### Before running
1. **Runtime → Change runtime type → A100 GPU**
2. Upload `data.zip` to: `My Drive/WWR_Seg_Model/data.zip`
3. **Run cells in order**

Cell 1 loads the code automatically:
- from Drive (if project folder exists), or
- **GitHub ZIP download** (if git clone fails), or
- `code.zip` on Drive (optional backup)

In [ ]:
# Install dependencies
!pip install -q tensorflow>=2.15 numpy pandas matplotlib scikit-learn openpyxl

## 1. Mount Drive & Load Project Code (required — run before any import)

In [ ]:
import os
import sys
import subprocess
import shutil
import urllib.request
import zipfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/kermanimohammad/U-Net_Segmentation.git"
GITHUB_ZIP = "https://github.com/kermanimohammad/U-Net_Segmentation/archive/refs/heads/main.zip"
LOCAL_REPO = Path("/content/U-Net_Segmentation")
DRIVE_BASE = Path("/content/drive/MyDrive/WWR_Seg_Model")

# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# ── Search for project on Drive or /content ──────────────────────────────────
def _find_project_root() -> Path | None:
    candidates = [
        LOCAL_REPO,
        DRIVE_BASE / "code",
        DRIVE_BASE / "U-Net_Segmentation",
        DRIVE_BASE,
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "WWR_Segmentation" / "__init__.py").exists():
            return candidate.resolve()
    return None


def _download_from_github_zip(target: Path) -> Path:
    """Download repo as ZIP (works when git clone fails in Colab)."""
    zip_path = Path("/content/repo_main.zip")
    print(f"Downloading from GitHub ZIP: {GITHUB_ZIP}")
    urllib.request.urlretrieve(GITHUB_ZIP, zip_path)

    if target.exists():
        shutil.rmtree(target)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")

    extracted = Path("/content/U-Net_Segmentation-main")
    shutil.move(str(extracted), str(target))
    zip_path.unlink(missing_ok=True)
    return target


def _extract_code_zip_from_drive() -> Path | None:
    """Extract My Drive/WWR_Seg_Model/code.zip if uploaded."""
    code_zip = DRIVE_BASE / "code.zip"
    if not code_zip.exists():
        return None

    print(f"Found code.zip on Drive — extracting to {LOCAL_REPO} ...")
    if LOCAL_REPO.exists():
        shutil.rmtree(LOCAL_REPO)
    LOCAL_REPO.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(code_zip, "r") as zf:
        zf.extractall(LOCAL_REPO.parent)

    # Handle zip that contains top-level U-Net_Segmentation/ or WWR_Segmentation/
    if (LOCAL_REPO / "WWR_Segmentation").exists():
        return LOCAL_REPO
    nested = LOCAL_REPO.parent / "U-Net_Segmentation"
    if nested.exists():
        if LOCAL_REPO.exists():
            shutil.rmtree(LOCAL_REPO)
        shutil.move(str(nested), str(LOCAL_REPO))
    return LOCAL_REPO if (LOCAL_REPO / "WWR_Segmentation").exists() else None


project_root = _find_project_root()

if project_root is None:
    project_root = _extract_code_zip_from_drive()

if project_root is None:
    print("Project not on Drive — trying git clone...")
    if LOCAL_REPO.exists():
        shutil.rmtree(LOCAL_REPO)
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(LOCAL_REPO)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("git clone failed:", result.stderr.strip())
        print("Trying GitHub ZIP download instead...")
        project_root = _download_from_github_zip(LOCAL_REPO)
    else:
        project_root = LOCAL_REPO

if not (project_root / "WWR_Segmentation" / "__init__.py").exists():
    raise FileNotFoundError(
        "WWR_Segmentation not found.\n"
        "Fix options:\n"
        "  1. Re-run this cell (ZIP download usually works)\n"
        "  2. Upload code.zip to My Drive/WWR_Seg_Model/code.zip\n"
        "  3. Upload full project folder to My Drive/WWR_Seg_Model/"
    )

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root : {project_root}")
print(f"Package OK   : {(project_root / 'WWR_Segmentation' / '__init__.py').exists()}")

## 2. Colab Setup — Mount Drive & Unzip data.zip to Local Storage

In [ ]:
from WWR_Segmentation.colab_setup import setup_colab
from WWR_Segmentation.dataset import get_dataset_info

# clone_if_missing=False because cell 1 already loaded the project
config = setup_colab(force_unzip=False, clone_if_missing=False)

get_dataset_info(config)

## 3. Training

In [ ]:
from WWR_Segmentation.trainer import train

model, history = train(config)

## 4. Evaluation (Validation + Independent Test Set)

In [ ]:
from WWR_Segmentation.evaluate import run_evaluation

eval_results = run_evaluation(config)
print(f"Test mean IoU: {eval_results['test']['metrics']['mean_iou']['value']:.4f}")

## 5. Window-to-Wall Ratio (WWR)

In [ ]:
from WWR_Segmentation.wwr import run_wwr_analysis

wwr_df = run_wwr_analysis(config)
wwr_df.head()

## 6. Optional: 5-Fold Cross-Validation

In [ ]:
# Uncomment to run (train set only — test set is never touched)
# from WWR_Segmentation.cross_validation import run_cross_validation
# cv_summary = run_cross_validation(config)

## 7. Inference on Test Images

In [ ]:
from WWR_Segmentation.inference import run_inference

run_inference(config, input_dir=config.test_images_dir)